# BEM–KM-PIKFNN–PINN full-field comparison

Reads existing `field_results/*.npz` files without rerunning solvers. Rows: ux, uy, von Mises stress. Columns: BEM, KM-PIKFNN, PINN. Each row shares one color scale and a flat-ended colorbar aligned with its actual plot height. Display units remain mm and MPa. Change FILES and CFG in the first code cell. The final error/export cell is preserved.

Hole-edge display: a narrow masked collar is filled with nearest valid samples, then the exact circular hole is drawn over it. This removes contour-mask white fringes. This is visualization only; exports and L2 errors use the original fields and mask. Requires scipy as well as numpy/matplotlib.


## 1. Plot settings


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import Normalize
from matplotlib.ticker import ScalarFormatter

# In a notebook, paths are relative to its working directory.
BASE_DIR = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
RESULT_DIR = BASE_DIR / 'field_results'
FIGURE_DIR = BASE_DIR / 'figures'
FILES = {
    'BEM': RESULT_DIR / 'bem_fields.npz',
    'KM-PIKFNN': RESULT_DIR / 'km_pikfnn_fields.npz',
    'PINN': RESULT_DIR / 'pinn_fields.npz',
}
CFG = dict(figsize=(10.4,9.5), title_fontsize=17, row_fontsize=19,
           label_fontsize=12, tick_fontsize=11, colorbar_fontsize=11,
           cmap='coolwarm', levels=101, save_dpi=400,
           save_outputs=True, show=True)
COLOR_LIMIT_MODE = 'full'  # 'percentile' clips outliers; disclose if used.
ROBUST_PERCENTILE = 99.5
EXPORT_DPI = CFG['save_dpi']
mpl.rcParams.update({
    'font.family':'serif', 'font.serif':['Times New Roman','DejaVu Serif'],
    'mathtext.fontset':'stix', 'font.size':12,
    'axes.linewidth':.8, 'xtick.direction':'in', 'ytick.direction':'in',
    'pdf.fonttype':42, 'ps.fonttype':42,
})
if CFG['save_outputs']:
    FIGURE_DIR.mkdir(parents=True,exist_ok=True)


## 2. Load and verify the three standardized data files


In [ ]:
REQUIRED_KEYS = {
    "X", "Y", "valid_mask", "ux", "uy", "sxx", "syy", "sxy", "vm",
    "a", "L_plot", "grid_size", "displacement_unit", "stress_unit", "method",
}


def load_result(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing result file: {path}\n"
            "Run the corresponding solver notebook and its final export cell first."
        )
    with np.load(path, allow_pickle=False) as archive:
        missing = sorted(REQUIRED_KEYS.difference(archive.files))
        if missing:
            raise KeyError(f"{path.name} is missing keys: {', '.join(missing)}")
        result = {key: archive[key].copy() for key in archive.files}

    if result["displacement_unit"].item() != "m":
        raise ValueError(f"{path.name}: displacement must be stored in metres.")
    if result["stress_unit"].item() != "Pa":
        raise ValueError(f"{path.name}: stress must be stored in pascals.")

    shape = result["X"].shape
    if len(shape) != 2 or result["Y"].shape != shape:
        raise ValueError(f"{path.name}: X and Y must be matching two-dimensional grids.")
    for key in ["valid_mask", "ux", "uy", "sxx", "syy", "sxy", "vm"]:
        if result[key].shape != shape:
            raise ValueError(
                f"{path.name}: {key} has shape {result[key].shape}, expected {shape}."
            )
    return result


results = {label: load_result(path) for label, path in FILES.items()}
method_order = list(FILES)
reference = results[method_order[0]]

# A shared color scale is only meaningful when every method uses the same coordinates
# and physical mask. Stop immediately if one notebook exported a different grid.
for label in method_order[1:]:
    current = results[label]
    if current["X"].shape != reference["X"].shape:
        raise ValueError(f"{label}: grid shape is inconsistent with BEM.")
    if not np.allclose(current["X"], reference["X"], rtol=0.0, atol=1e-12):
        raise ValueError(f"{label}: X grid is inconsistent with BEM.")
    if not np.allclose(current["Y"], reference["Y"], rtol=0.0, atol=1e-12):
        raise ValueError(f"{label}: Y grid is inconsistent with BEM.")
    if not np.array_equal(current["valid_mask"], reference["valid_mask"]):
        raise ValueError(f"{label}: valid-domain mask is inconsistent with BEM.")
    if not np.isclose(float(current["a"]), float(reference["a"]), atol=1e-12):
        raise ValueError(f"{label}: hole radius is inconsistent with BEM.")

print("Loaded and verified:")
for label in method_order:
    data = results[label]
    params = int(data["trainable_parameters"]) if "trainable_parameters" in data else None
    elapsed = float(data["training_time_s"]) if "training_time_s" in data else np.nan
    details = []
    if params is not None and params >= 0:
        details.append(f"parameters={params}")
    if np.isfinite(elapsed):
        details.append(f"time={elapsed:.2f} s")
    suffix = f" ({', '.join(details)})" if details else ""
    print(f"  {label:11s}: {data['X'].shape[1]} x {data['X'].shape[0]}{suffix}")


## 3. Generate and export the 3 × 3 full-field figure


In [ ]:
# Preserve the original display units: displacement in mm, stress in MPa.
field_specs = [
    dict(key='ux', title=r'$u_x$', scale=1e3, symmetric=True, unit='Displacement (mm)'),
    dict(key='uy', title=r'$u_y$', scale=1e3, symmetric=True, unit='Displacement (mm)'),
    dict(key='vm', title=r'$\sigma_{\mathrm{VM}}$', scale=1e-6,
         symmetric=False, unit='Stress (MPa)'),
]
valid_mask = np.asarray(reference['valid_mask'],dtype=bool)
X,Y = (np.asarray(reference[k],dtype=float) for k in ('X','Y'))
a=float(reference['a'])
extent=[float(X.min()),float(X.max()),float(Y.min()),float(Y.max())]
if not np.isfinite(X).all() or not np.isfinite(Y).all():
    raise ValueError('X and Y must contain finite coordinates.')

def common_limits(spec):
    arrays=[]
    for label in method_order:
        values=np.asarray(results[label][spec['key']],dtype=float)*spec['scale']
        if not np.isfinite(values[valid_mask]).all():
            raise ValueError(f"{label}: non-finite {spec['key']} values inside the valid domain.")
        if not np.any(valid_mask):raise ValueError('The valid-domain mask is empty.')
        arrays.append(values[valid_mask])
    values=np.concatenate(arrays)
    if COLOR_LIMIT_MODE=='full':
        limit=float(np.max(np.abs(values))) if spec['symmetric'] else float(np.max(values))
    elif COLOR_LIMIT_MODE=='percentile':
        limit=float(np.percentile(np.abs(values) if spec['symmetric'] else values,
                                  ROBUST_PERCENTILE))
    else:raise ValueError("COLOR_LIMIT_MODE must be 'full' or 'percentile'.")
    limit=max(limit,1e-30)
    return (-limit,limit) if spec['symmetric'] else (min(0.,float(values.min())),limit)

limits=[common_limits(spec) for spec in field_specs]
fig=plt.figure(figsize=CFG['figsize'],dpi=150)
grid=fig.add_gridspec(3,3,left=.12,right=.865,bottom=.065,top=.945,
                      wspace=.14,hspace=.16)
axes=np.array([[fig.add_subplot(grid[r,c]) for c in range(3)] for r in range(3)])
mappables=[]
for row,spec in enumerate(field_specs):
    vmin,vmax=limits[row]
    norm=Normalize(vmin=vmin,vmax=vmax)
    levels=np.linspace(vmin,vmax,CFG['levels'])
    for column,(ax,label) in enumerate(zip(axes[row],method_order)):
        values=np.asarray(results[label][spec['key']],dtype=float)*spec['scale']
        # The exact circular patch covers extrapolated interior values.
        # Fill only a one-cell collar at the circular hole for display.
        # No high-resolution physical data are invented; numerical errors use originals.
        from scipy.ndimage import distance_transform_edt
        dx=float(np.median(np.abs(np.diff(X,axis=1))))
        dy=float(np.median(np.abs(np.diff(Y,axis=0))))
        collar=1.05*np.hypot(dx,dy)
        distance,nearest=distance_transform_edt(~valid_mask,sampling=(dy,dx),return_indices=True)
        radius=np.hypot(X,Y)
        hole_fill=(~valid_mask)&(radius<=a+collar)
        # Ensure an exterior collar is at most roughly one diagonal grid step wide.
        hole_fill &= (radius<a)|(distance<=collar)
        display_values=values.copy()
        display_values[hole_fill]=values[tuple(nearest[:,hole_fill])]
        plotted=np.ma.masked_where(~(valid_mask|hole_fill),display_values)
        if COLOR_LIMIT_MODE=='percentile':plotted=np.ma.clip(plotted,vmin,vmax)
        im=ax.contourf(X,Y,plotted,levels=levels,norm=norm,
                       cmap=CFG['cmap'],extend='neither',corner_mask=True)
        ax.add_patch(Circle((0.,0.),a,facecolor='white',edgecolor='black',
                            linewidth=.8,zorder=5))
        ax.set_aspect('equal',adjustable='box')
        ax.set_xlim(extent[:2]);ax.set_ylim(extent[2:])
        ax.set_xticks(np.linspace(extent[0],extent[1],3))
        ax.set_yticks(np.linspace(extent[2],extent[3],3))
        ax.tick_params(direction='in',top=True,right=True,
                       labelsize=CFG['tick_fontsize'],
                       labelbottom=(row==2),labelleft=(column==0))
        for spine in ax.spines.values():
            spine.set_color('#666666');spine.set_linewidth(.8)
        if column==0:ax.set_ylabel(r'$y$ (m)',fontsize=CFG['label_fontsize'])
        if row==2:ax.set_xlabel(r'$x$ (m)',fontsize=CFG['label_fontsize'])
        if row==0:ax.set_title(label,fontsize=CFG['title_fontsize'],pad=11)
    mappables.append(im)

# Apply aspect ratios BEFORE measuring the actual plot boxes.
fig.canvas.draw()
color_axes=[]
for row,(spec,im) in enumerate(zip(field_specs,mappables)):
    box=axes[row,2].get_position()
    cax=fig.add_axes([box.x1+.018,box.y0,.017,box.height])
    color_axes.append(cax)
    cb=fig.colorbar(im,cax=cax,ticks=np.linspace(im.norm.vmin,im.norm.vmax,5))
    formatter=ScalarFormatter(useMathText=True);formatter.set_powerlimits((-2,2))
    cb.formatter=formatter;cb.update_ticks()
    cb.ax.tick_params(labelsize=CFG['colorbar_fontsize'],length=3)
    cb.ax.yaxis.get_offset_text().set_fontsize(CFG['colorbar_fontsize'])
    cb.set_label(spec['unit'],fontsize=CFG['label_fontsize'],labelpad=6)
    left=axes[row,0].get_position()
    fig.text(.025,(left.y0+left.y1)/2,spec['title'],rotation=90,
             va='center',ha='center',fontsize=CFG['row_fontsize'])

mode_suffix='' if COLOR_LIMIT_MODE=='full' else f'_p{ROBUST_PERCENTILE:g}'
png_file=FIGURE_DIR/f'full_field_comparison_3x3{mode_suffix}.png'
pdf_file=FIGURE_DIR/f'full_field_comparison_3x3{mode_suffix}.pdf'
tiff_file=FIGURE_DIR/f'full_field_comparison_3x3{mode_suffix}.tiff'
if CFG['save_outputs']:
    for path in (png_file,pdf_file,tiff_file):
        kwargs=dict(dpi=EXPORT_DPI,bbox_inches='tight',facecolor='white')
        if path.suffix=='.tiff':kwargs['pil_kwargs']={'compression':'tiff_lzw'}
        fig.savefig(path,**kwargs);print('Saved:',path.resolve())
if CFG['show']:plt.show()
print('Shared color limits:')
for spec,(vmin,vmax) in zip(field_specs,limits):
    print(f"  {spec['title']} ({spec['unit']}): [{vmin:.6g}, {vmax:.6g}]")


## Figure caption template

**Fig. X.** Full-field comparison for the pressurized circular-hole problem. Rows: displacement components $u_x$ and $u_y$, and von Mises stress $\sigma_{\mathrm{VM}}$; columns: BEM, KM-PIKFNN, and PINN. Each row uses a common color scale. Displacements and stresses are shown in mm and MPa, respectively.


In [ ]:
import csv

comparison_methods = ["PINN", "KM-PIKFNN"]
reference_method = "BEM"
reference_data = results[reference_method]


def relative_l2(prediction, reference_values, domain_mask, field_name):
    """Discrete relative L2 error on common finite exterior grid points."""
    prediction = np.asarray(prediction, dtype=float)
    reference_values = np.asarray(reference_values, dtype=float)
    active = (
        np.asarray(domain_mask, dtype=bool)
        & np.isfinite(prediction)
        & np.isfinite(reference_values)
    )
    if not np.any(active):
        raise ValueError(f"No common finite points are available for {field_name}.")

    denominator = np.linalg.norm(reference_values[active].reshape(-1), ord=2)
    if denominator <= np.finfo(float).eps:
        raise ZeroDivisionError(
            f"The BEM L2 norm of {field_name} is zero; relative error is undefined."
        )
    numerator = np.linalg.norm(
        (prediction[active] - reference_values[active]).reshape(-1), ord=2
    )
    return float(numerator / denominator), int(np.count_nonzero(active))


error_summary = {}
for method in comparison_methods:
    method_data = results[method]
    ux_error, n_ux = relative_l2(
        method_data["ux"], reference_data["ux"], valid_mask, "ux"
    )
    uy_error, n_uy = relative_l2(
        method_data["uy"], reference_data["uy"], valid_mask, "uy"
    )
    vm_error, n_vm = relative_l2(
        method_data["vm"], reference_data["vm"], valid_mask, "von Mises stress"
    )
    error_summary[method] = {
        "ux": ux_error,
        "uy": uy_error,
        "vm": vm_error,
        "n_ux": n_ux,
        "n_uy": n_uy,
        "n_vm": n_vm,
    }

# Print a compact table suitable for checking values before copying them into the paper.
print("Relative L2 errors with respect to BEM")
print("-" * 66)
print(f"{'Method':<14s} {'Ux':>15s} {'Uy':>15s} {'von Mises':>15s}")
print("-" * 66)
for method in comparison_methods:
    row = error_summary[method]
    print(
        f"{method:<14s} {row['ux']:>15.6e} {row['uy']:>15.6e} "
        f"{row['vm']:>15.6e}"
    )
print("-" * 66)

# Save exact numerical values for the manuscript table.
csv_file = FIGURE_DIR / "relative_l2_errors_vs_bem.csv"
with csv_file.open("w", newline="", encoding="utf-8-sig") as stream:
    writer = csv.DictWriter(
        stream,
        fieldnames=[
            "method",
            "ux_relative_l2",
            "uy_relative_l2",
            "von_mises_relative_l2",
            "evaluation_points",
        ],
    )
    writer.writeheader()
    for method in comparison_methods:
        row = error_summary[method]
        writer.writerow({
            "method": method,
            "ux_relative_l2": f"{row['ux']:.12e}",
            "uy_relative_l2": f"{row['uy']:.12e}",
            "von_mises_relative_l2": f"{row['vm']:.12e}",
            "evaluation_points": min(row["n_ux"], row["n_uy"], row["n_vm"]),
        })

# Optional publication-quality grouped bar chart. A logarithmic y-axis is used because
# PINN and KM-PIKFNN errors may differ by orders of magnitude.
quantity_keys = ["ux", "uy", "vm"]
quantity_labels = [r"$u_x$", r"$u_y$", r"$\sigma_{\mathrm{vM}}$"]
x_positions = np.arange(len(quantity_keys), dtype=float)
bar_width = 0.34
# Muted NPG-inspired colors remain distinct in print without appearing overly bright.
bar_colors = {"PINN": "#3C5488", "KM-PIKFNN": "#E64B35"}

error_fig, error_ax = plt.subplots(figsize=(6.8, 4.2), constrained_layout=True)
for method_index, method in enumerate(comparison_methods):
    exact_values = np.array(
        [error_summary[method][key] for key in quantity_keys], dtype=float
    )
    plotted_values = np.maximum(exact_values, 1.0e-16)
    offset = (method_index - 0.5) * bar_width
    bars = error_ax.bar(
        x_positions + offset,
        plotted_values,
        width=bar_width,
        label=method,
        color=bar_colors[method],
        edgecolor="black",
        linewidth=0.55,
        zorder=3,
    )
    for bar, exact_value in zip(bars, exact_values):
        error_ax.annotate(
            f"{exact_value:.1e}",
            xy=(bar.get_x() + bar.get_width() / 2.0, bar.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7,
            rotation=90,
        )

error_ax.set_yscale("log")
error_ax.set_ylabel(r"Relative $L_2$ error")
error_ax.set_xticks(x_positions, quantity_labels)
# Display major ticks explicitly as 10^{-1}, 10^{-2}, 10^{-3}, etc.
error_ax.yaxis.set_major_locator(mpl.ticker.LogLocator(base=10.0, numticks=10))
error_ax.yaxis.set_major_formatter(mpl.ticker.LogFormatterMathtext(base=10.0))
error_ax.yaxis.set_minor_locator(
    mpl.ticker.LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=100)
)
error_ax.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

# Expand the vertical range to complete powers of ten for clean scientific notation.
all_positive_errors = np.array([
    error_summary[method][key]
    for method in comparison_methods
    for key in quantity_keys
    if error_summary[method][key] > 0.0
])
if all_positive_errors.size:
    lower_decade = np.floor(np.log10(all_positive_errors.min())) - 1.0
    upper_decade = np.ceil(np.log10(all_positive_errors.max()))
    error_ax.set_ylim(10.0 ** lower_decade, 10.0 ** upper_decade)

error_ax.legend(
    frameon=True,
    fancybox=False,
    edgecolor="0.25",
    facecolor="white",
    framealpha=0.95,
    ncol=1,
    loc="upper right",
)
error_ax.grid(axis="y", which="both", linestyle="--", linewidth=0.5, alpha=0.45)
error_ax.set_axisbelow(True)

error_png = FIGURE_DIR / "relative_l2_errors_vs_bem.png"
error_pdf = FIGURE_DIR / "relative_l2_errors_vs_bem.pdf"
error_fig.savefig(error_png, dpi=EXPORT_DPI, bbox_inches="tight", facecolor="white")
error_fig.savefig(error_pdf, dpi=EXPORT_DPI, bbox_inches="tight", facecolor="white")
plt.show()

print(f"\nCSV saved to: {csv_file.resolve()}")
print(f"Error chart saved to: {error_png.resolve()}")
print(f"Error chart saved to: {error_pdf.resolve()}")
